# ♻️ Smart E-Waste Segregation System: YOLOv8 Training & Inference
**Project:** Conveyor-Based Automated Electronic Waste Segregation using YOLOv8 & 6-DOF Robotic Arm  
**GitHub Repository:** [neilgibsonpinto-art/ewaste](https://github.com/neilgibsonpinto-art/ewaste)  
**Target Classes:** `0: pcb`, `1: battery`, `2: display`  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neilgibsonpinto-art/ewaste/blob/main/Smart_EWaste_Colab.ipynb)

## 1. Setup Environment & Clone Project Repository
Clone the complete project repository from GitHub and install `ultralytics`, `roboflow`, and dependencies.

In [ ]:
# Clone the repository from GitHub
!git clone https://github.com/neilgibsonpinto-art/ewaste.git
%cd ewaste

# Install YOLOv8 and computer vision packages
!pip install -q ultralytics roboflow opencv-python matplotlib Pillow

import torch
import ultralytics
print('PyTorch Version:', torch.__version__)
print('Ultralytics YOLO Version:', ultralytics.__version__)
print('GPU Available:', torch.cuda.is_available())

## 2. Dataset Inspection & Preview
Inspect `data.yaml` and visualize training samples with bounding boxes.

In [ ]:
import yaml
import glob
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Read and configure data.yaml for Google Colab
with open('dataset/data.yaml', 'r') as f:
    data_config = yaml.safe_load(f)

data_config['path'] = '/content/ewaste/dataset'
with open('dataset/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print('Dataset Configuration:')
print(yaml.dump(data_config))

# Preview 4 random training images
train_images = glob.glob('dataset/images/train/*.jpg')[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img_p in zip(axes, train_images):
    ax.imshow(Image.open(img_p))
    ax.set_title(Path(img_p).stem)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Train Lightweight YOLOv8-Nano Model
Train the model for 30 epochs with free GPU acceleration on Google Colab.

In [ ]:
from ultralytics import YOLO

# Load pre-trained nano weights
model = YOLO('yolov8n.pt')

# Train on E-Waste dataset
results = model.train(
    data='dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='runs/train',
    name='ewaste_colab_yolov8n',
    plots=True,
    verbose=True
)

print('Training Complete!')
print('Best weights saved at: runs/train/ewaste_colab_yolov8n/weights/best.pt')

## 4. Visualize Model Performance Curves
Display loss curves, confusion matrix, and Precision-Recall (PR) curves.

In [ ]:
from IPython.display import Image as IPyImage, display

print('--- Training Loss & Validation Results ---')
display(IPyImage('runs/train/ewaste_colab_yolov8n/results.png'))

print('--- Confusion Matrix ---')
display(IPyImage('runs/train/ewaste_colab_yolov8n/confusion_matrix.png'))

print('--- Precision-Recall Curve ---')
display(IPyImage('runs/train/ewaste_colab_yolov8n/BoxPR_curve.png'))

## 5. Test Inference on Test Images with Live Counter
Run detection on validation images, count sorted parts, and visualize predictions.

In [ ]:
import glob
from collections import Counter

# Load the trained model
trained_model = YOLO('runs/train/ewaste_colab_yolov8n/weights/best.pt')

# Run inference on validation images
val_images = glob.glob('dataset/images/val/*.jpg')[:6]
counter = Counter()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_p in zip(axes.flatten(), val_images):
    res = trained_model.predict(img_p, conf=0.50, verbose=False)[0]
    
    # Count classes detected
    for box in res.boxes:
        cls_name = res.names[int(box.cls[0])]
        counter[cls_name] += 1
        
    # Plot annotated image (RGB)
    annotated_rgb = res.plot()[:, :, ::-1]
    ax.imshow(annotated_rgb)
    ax.set_title(Path(img_p).stem)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('=' * 40)
print('  SEGREGATION TALLY SUMMARY')
print('=' * 40)
for k, v in counter.items():
    print(f'  {k.upper()}: {v} item(s)')
print(f'  TOTAL PROCESSED: {sum(counter.values())}')
print('=' * 40)

## 6. Export Model for Edge Devices (Raspberry Pi 5 / Laptop)
Export weights to ONNX format for ultra-fast edge deployment, and download the `.pt` file.

In [ ]:
from google.colab import files

# Export to ONNX
!yolo export model=runs/train/ewaste_colab_yolov8n/weights/best.pt format=onnx imgsz=640

# Download best.pt directly to your computer
files.download('runs/train/ewaste_colab_yolov8n/weights/best.pt')
print('Model weights ready for edge deployment!')